# XCoder-0.8B - Fine-tune di Kaggle (T4 x2)

**Panduan**
1. Settings -> Accelerator: **GPU T4 x2**; **Internet: ON**.
2. Buat **Kaggle Dataset private bernama `xcoder-training`** (New Dataset -> upload, versi baharu bila data berubah):
   - `all.jsonl` (dari repo: `datasets/all.jsonl` - pastikan fable_ms sudah dimerge)
   - `train_unsloth.py` (dari repo: `finetune/train_unsloth.py`)
3. Jalankan sel dari atas ke bawah. Sel install gantikan torch bawaan Kaggle dengan pin yang diuji (~5-10 min).
   Checkpoint setiap 100 langkah ke `/kaggle/working/xcoder-best-0.8b/` (kekal antara sesi; Save Version untuk snapshot).
4. Terputus? Run semula - auto-resume dari checkpoint terakhir.
5. Eksport tak sempat? Run sel `--export-only` sahaja.
6. Nota: P100 tak sokong bf16 - kalau ada isu, guna T4 x2.

Anggaran: ~2.5-4.5 jam (1 epoch, packing, max_seq_length=3072).

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil

SRC = '/kaggle/input/xcoder-training'
WORK = '/kaggle/working'
if not os.path.isdir(SRC):
    raise SystemExit(f'{SRC} tiada - buat Kaggle Dataset "xcoder-training" dengan '
                     'all.jsonl + train_unsloth.py.')
for name in ('train_unsloth.py', 'all.jsonl'):
    shutil.copy(f'{SRC}/{name}', f'{WORK}/{name}')
    print(name, round(os.path.getsize(f'{WORK}/{name}') / 1e6, 1), 'MB')

In [ ]:
# Resipi rasmi Unsloth Qwen3.5 - pins tepat (sama seperti Colab).
# Gantikan torch bawaan Kaggle; ambil ~5-10 min.
# Jika import unsloth gagal selepas ini: Run -> Restart session, teruskan dari sel seterusnya.
!pip install --upgrade -qqq uv
!uv pip install -qqq \
    "torch==2.8.0" "triton>=3.3.0" torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq --no-deps "torchcodec==0.7.0"
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
print('install selesai')

In [ ]:
%cd /kaggle/working
!python train_unsloth.py --data all --out /kaggle/working/xcoder-best-0.8b

In [ ]:
# Jalankan SAHAJA jika training sudah selesai tetapi eksport merged/GGUF tidak sempat
%cd /kaggle/working
!python train_unsloth.py --export-only --out /kaggle/working/xcoder-best-0.8b

In [ ]:
import glob, os, shutil

gs = glob.glob('/kaggle/working/xcoder-best-0.8b-gguf/*.gguf')
if not gs:
    print('Tiada GGUF dijumpai - pastikan training + eksport selesai (sel sebelumnya)')
for f in gs:
    dst = '/kaggle/working/xcoder-best-0.8b-q4_k_m.gguf'
    shutil.copy(f, dst)
    print('sedia:', dst, round(os.path.getsize(dst) / 1e6), 'MB')
print('Muat turun dari tab Output (Save Version dahulu).')

## Selepas ini (di PC)
1. Muat turun `xcoder-best-0.8b-q4_k_m.gguf`.
2. Daftar: `/model add <fail> --name xcoder-best --domains general`
3. Uji identiti: tanya "Siapa anda?" / "Are you Qwen?" -> "Saya XCoder..." tanpa sebut Qwen/Alibaba.